# Notebook 2: Data Preprocessing & Feature Engineering

This step combines initial data cleaning, feature engineering, and data encoding to prepare the dataset for Machine Learning models.

In [1]:
# Load the dataset
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/BankChurners.csv')
cols_to_drop = [
    'CLIENTNUM',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'
]
df = df.drop(columns=cols_to_drop)
print('size of dataset: ', df.shape)
df.nunique()

size of dataset:  (10127, 20)


Attrition_Flag                 2
Customer_Age                  45
Gender                         2
Dependent_count                6
Education_Level                7
Marital_Status                 4
Income_Category                6
Card_Category                  4
Months_on_book                44
Total_Relationship_Count       6
Months_Inactive_12_mon         7
Contacts_Count_12_mon          7
Credit_Limit                6205
Total_Revolving_Bal         1974
Avg_Open_To_Buy             6813
Total_Amt_Chng_Q4_Q1        1158
Total_Trans_Amt             5033
Total_Trans_Ct               126
Total_Ct_Chng_Q4_Q1          830
Avg_Utilization_Ratio        964
dtype: int64

## Explore Categorical Variables
Check the number and distribution of unique values for string (object) columns.

In [2]:
object_cols = df.select_dtypes(include=['object']).columns
print("Object columns:", list(object_cols))
print("-" * 40)

for col in object_cols:
    print(f'Number of unique values for column {col}: {df[col].nunique()}')
    print(df[col].value_counts())
    print("-" * 40)

Object columns: ['Attrition_Flag', 'Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']
----------------------------------------
Number of unique values for column Attrition_Flag: 2
Attrition_Flag
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64
----------------------------------------
Number of unique values for column Gender: 2
Gender
F    5358
M    4769
Name: count, dtype: int64
----------------------------------------
Number of unique values for column Education_Level: 7
Education_Level
Graduate         3128
High School      2013
Unknown          1519
Uneducated       1487
College          1013
Post-Graduate     516
Doctorate         451
Name: count, dtype: int64
----------------------------------------
Number of unique values for column Marital_Status: 4
Marital_Status
Married     4687
Single      3943
Unknown      749
Divorced     748
Name: count, dtype: int64
----------------------------------------
Number of unique valu

## Feature Engineering
1. **Group Rare Categories:** The `Platinum` card has very few instances (only 20). We group it with `Gold` to prevent noise.
2. **Feature Creation:** Create `Avg_Amt_Per_Trans` (Average amount per transaction).
3. **Handle Multi-collinearity:** Drop `Avg_Open_To_Buy` (redundant information), KEEP `Credit_Limit`.

In [3]:
# Group Platinum and Gold cards
df['Card_Category'] = df['Card_Category'].replace({'Platinum': 'Gold'})

# Create a new feature: Average amount per transaction
df['Avg_Amt_Per_Trans'] = df['Total_Trans_Amt'] / df['Total_Trans_Ct']

# Drop Avg_Open_To_Buy due to 99.6% correlation with Credit_Limit
df.drop(columns=['Avg_Open_To_Buy'], inplace=True)

print('Dataset size after Feature Engineering:', df.shape)

Dataset size after Feature Engineering: (10127, 20)


## Data Encoding
Instead of using `LabelEncoder` mechanically for all categorical columns (which assigns numbers alphabetically and ruins the natural order, e.g., making 'Uneducated' > 'Doctorate'), we apply two specific strategies:
1. **Ordinal Encoding (Custom Mapping):** For features with a natural hierarchy (`Education_Level`, `Income_Category`, `Card_Category`) so the model understands the increasing order.
2. **One-Hot Encoding:** For nominal features without any hierarchy (`Gender`, `Marital_Status`). We use `drop_first=True` to avoid the Dummy Variable Trap (multi-collinearity).

In [4]:
# Ordinal Encoding (Custom Mapping)
edu_map = {
    'Uneducated': 0, 
    'High School': 1, 
    'College': 2, 
    'Graduate': 3, 
    'Post-Graduate': 4, 
    'Doctorate': 5, 
    'Unknown': -1  # Keep 'Unknown' separate from the hierarchy
}

income_map = {
    'Less than $40K': 0, 
    '$40K - $60K': 1, 
    '$60K - $80K': 2, 
    '$80K - $120K': 3, 
    '$120K +': 4, 
    'Unknown': -1
}

card_map = {
    'Blue': 0, 
    'Silver': 1, 
    'Gold': 2  # Platinum was already grouped into Gold earlier
}

# Apply mapping to the dataframe
df['Education_Level'] = df['Education_Level'].map(edu_map)
df['Income_Category'] = df['Income_Category'].map(income_map)
df['Card_Category'] = df['Card_Category'].map(card_map)

# One-Hot Encoding (Create dummy variables for nominal features)
df = pd.get_dummies(df, columns=['Gender', 'Marital_Status'], drop_first=True)

print(f"[*] Dataset shape after proper encoding: {df.shape}")
display(df.head())

[*] Dataset shape after proper encoding: (10127, 22)


,Attrition_Flag,Customer_Age,Dependent_count,Education_Level,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,...,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Avg_Amt_Per_Trans,Gender_M,Marital_Status_Married,Marital_Status_Single,Marital_Status_Unknown
0,Existing Customer,45,3,1,2,0,39,5,1,3,...,1.335,1144,42,1.625,0.061,27.238095,True,True,False,False
1,Existing Customer,49,5,3,0,0,44,6,1,2,...,1.541,1291,33,3.714,0.105,39.121212,False,False,True,False
2,Existing Customer,51,3,3,3,0,36,4,1,0,...,2.594,1887,20,2.333,0.000,94.350000,True,True,False,False
3,Existing Customer,40,4,1,0,0,34,3,4,1,...,1.405,1171,20,2.333,0.760,58.550000,False,False,False,True
4,Existing Customer,40,3,0,2,0,21,5,1,0,...,2.175,816,28,2.500,0.000,29.142857,True,True,False,False


## Train/Test Split, Scale & SMOTE
Correct sequence to prevent Data Leakage:
1. Split Train/Test sets (with `stratify=y` to maintain the churn ratio).
2. Scale the Train and Test sets (`StandardScaler`).
3. Apply SMOTE **only on the scaled Train set**.

In [5]:
!pip install imbalanced-learn -q

In [5]:
# Splitting input/output
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

X = df.drop(['Attrition_Flag'], axis=1)
y = df['Attrition_Flag']

# Train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

# Scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Data balancing
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Original Train shape: {X_train.shape}")
print(f"Train shape after SMOTE: {X_train_balanced.shape} (Balanced classes)")
print(f"Test shape: {X_test_scaled.shape}")

Original Train shape: (8101, 21)
Train shape after SMOTE: (13598, 21) (Balanced classes)
Test shape: (2026, 21)



Experiment: Comparing Data Balancing Techniques
We will train a baseline Random Forest model using 3 different balancing strategies to see which one performs best at detecting churned customers (Class 1).
1. **SMOTE:** Synthetic data generation.
2. **Random Under-Sampling (RUS):** Reducing the majority class.
3. **Class Weights:** No data modification, algorithm penalizes minority misclassification.
4. **SMOTE-ENN:** Combining the SMOTE method and Edited Nearest Neighbor (ENN)

In [6]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.combine import SMOTEENN

# Prepare Random Under-Sampling data
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)

# Prepare SMOTE-ENN data
smote_enn = SMOTEENN(random_state=42)
X_train_smenn, y_train_smenn = smote_enn.fit_resample(X_train_scaled, y_train)

# Print shapes to verify
print(f"Original Shape (for Class Weights): {X_train_scaled.shape}")
print(f"Shape after SMOTE: {X_train_balanced.shape}")
print(f"Shape after Under-Sampling: {X_train_rus.shape}")
print(f"Shape after SMOTE-ENN: {X_train_smenn.shape}")
print("(Notice how the size is smaller than pure SMOTE because ENN deleted the noisy points)\n")

# Define a function to train and evaluate
def evaluate_balancing_method(X_tr, y_tr, method_name, use_class_weight=False):
    # Initialize Random Forest
    if use_class_weight:
        # 'balanced' automatically adjusts weights inversely proportional to class frequencies
        model = RandomForestClassifier(random_state=42, class_weight='balanced')
    else:
        model = RandomForestClassifier(random_state=42)
        
    # Train model
    model.fit(X_tr, y_tr)
    
    # Predict on the UNSEEN test set
    y_pred = model.predict(X_test_scaled)
    
    # Print results
    print(f'{"="*40}')
    print(f"Results for: {method_name}")
    print(f'{"="*40}')
    print(classification_report(y_test, y_pred))
    print("\n")

# Run the Experiment
evaluate_balancing_method(X_train_balanced, y_train_balanced, "1. SMOTE")
evaluate_balancing_method(X_train_rus, y_train_rus, "2. RANDOM UNDER-SAMPLING")
evaluate_balancing_method(X_train_scaled, y_train, "3. CLASS WEIGHTS (Original Data)", use_class_weight=True)
evaluate_balancing_method(X_train_smenn, y_train_smenn, "4. SMOTE-ENN")

Original Shape (for Class Weights): (8101, 21)
Shape after SMOTE: (13598, 21)
Shape after Under-Sampling: (2604, 21)
Shape after SMOTE-ENN: (12243, 21)
(Notice how the size is smaller than pure SMOTE because ENN deleted the noisy points)

Results for: 1. SMOTE
                   precision    recall  f1-score   support

Attrited Customer       0.86      0.91      0.89       325
Existing Customer       0.98      0.97      0.98      1701

         accuracy                           0.96      2026
        macro avg       0.92      0.94      0.93      2026
     weighted avg       0.96      0.96      0.96      2026



Results for: 2. RANDOM UNDER-SAMPLING
                   precision    recall  f1-score   support

Attrited Customer       0.72      0.95      0.82       325
Existing Customer       0.99      0.93      0.96      1701

         accuracy                           0.93      2026
        macro avg       0.86      0.94      0.89      2026
     weighted avg       0.95      0.93      0

With SMOTE, the highest balanced F1 score (0.89) is achieved while simultaneously optimizing both the cost problem (Precision 0.86) and retention efficiency (Recall 0.91).